# Home Assignment: Dimensionality Reduction in Chemistry (PCA, t-SNE, UMAP)



## Setup
Run this cell first. It installs RDKit and UMAP

In [ ]:
!pip install -q rdkit umap-learn

---
# Part A

In [ ]:
import numpy as np

## A1. PCA on two descriptors

Four molecules are described by two descriptors measured on the same scale, $(x_1, x_2)$:

$P = (7, 3), \quad Q = (1, 1), \quad R = (5, 3), \quad S = (3, 1)$

**(a)** Center the data and compute the covariance matrix using $1/N$ normalization.

**(b)** Find both eigenvalues and the corresponding unit eigenvectors. Verify that the eigenvalues sum to the trace of the covariance matrix.

**(c)** What fraction of the total variance is explained by PC1? What angle does PC1 make with the $x_1$ axis?

**(d)** Compute the PC1 scores of all four molecules. Show that the mean squared reconstruction error, when only PC1 is kept, equals the discarded eigenvalue.

**Your answer:**



In [4]:
import numpy as np

X = np.array([[7, 3], [1, 1], [5, 3], [3, 1]], dtype=float)

# (a) Center the data and compute the covariance matrix
X_centered = X - np.mean(X, axis=0)
cov_matrix = np.cov(X_centered, rowvar=False, bias=True) # bias=True for 1/N normalization

print("Centered data X_centered:")
print(X_centered)
print("\nCovariance matrix (1/N normalization):")
print(cov_matrix)

# (b) Find both eigenvalues and the corresponding unit eigenvectors.
# Verify that the eigenvalues sum to the trace of the covariance matrix.
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Sort eigenvalues and eigenvectors in descending order
sorted_indices = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_indices]
eigenvectors = eigenvectors[:, sorted_indices]

# Calculate the trace of the covariance matrix
trace_cov_matrix = np.trace(cov_matrix)

print("\nEigenvalues:")
print(eigenvalues)
print("\nEigenvectors:")
print(eigenvectors)
print(f"\nSum of eigenvalues: {np.sum(eigenvalues):.2f}")
print(f"Trace of covariance matrix: {trace_cov_matrix:.2f}")
print(f"Sum of eigenvalues equals trace: {np.isclose(np.sum(eigenvalues), trace_cov_matrix)}")

# (c) What fraction of the total variance is explained by PC1? What angle does PC1 make with the x1 axis?

total_variance = np.sum(eigenvalues)
explained_variance_pc1 = eigenvalues[0]
fraction_explained_pc1 = explained_variance_pc1 / total_variance

# PC1 is the first eigenvector (corresponding to the largest eigenvalue)
pc1_vector = eigenvectors[:, 0]

# Angle with x1 axis (in radians then degrees)
angle_rad = np.arctan2(pc1_vector[1], pc1_vector[0])
angle_deg = np.degrees(angle_rad)

print(f"\nFraction of total variance explained by PC1: {fraction_explained_pc1:.4f}")
print(f"Angle PC1 makes with the x1 axis: {angle_deg:.2f} degrees")

# (d) Compute the PC1 scores of all four molecules. Show that the mean squared reconstruction error,
# when only PC1 is kept, equals the discarded eigenvalue.

# PC1 scores: Project the centered data onto the first eigenvector
pc1_scores = X_centered @ pc1_vector
print("\nPC1 Scores:")
print(pc1_scores)

# Reconstruct data using only PC1
X_reconstructed_pc1 = np.outer(pc1_scores, pc1_vector)

# Mean squared reconstruction error
mse = np.mean(np.sum((X_centered - X_reconstructed_pc1)**2, axis=1))

# The discarded eigenvalue is the second eigenvalue (eigenvalues[1])
discarded_eigenvalue = eigenvalues[1]

print(f"\nMean Squared Reconstruction Error (using only PC1): {mse:.4f}")
print(f"Discarded Eigenvalue (eigenvalues[1]): {discarded_eigenvalue:.4f}")
print(f"MSE equals discarded eigenvalue: {np.isclose(mse, discarded_eigenvalue)}")

Centered data X_centered:
[[ 3.  1.]
 [-3. -1.]
 [ 1.  1.]
 [-1. -1.]]

Covariance matrix (1/N normalization):
[[5. 2.]
 [2. 1.]]

Eigenvalues:
[5.82842712 0.17157288]

Eigenvectors:
[[ 0.92387953 -0.38268343]
 [ 0.38268343  0.92387953]]

Sum of eigenvalues: 6.00
Trace of covariance matrix: 6.00
Sum of eigenvalues equals trace: True

Fraction of total variance explained by PC1: 0.9714
Angle PC1 makes with the x1 axis: 22.50 degrees

PC1 Scores:
[ 3.15432203 -3.15432203  1.30656296 -1.30656296]

Mean Squared Reconstruction Error (using only PC1): 0.1716
Discarded Eigenvalue (eigenvalues[1]): 0.1716
MSE equals discarded eigenvalue: True


## A2. t-SNE conditional probabilities

A molecule $i$ has three neighbors at distances 1, 1, and 2 in descriptor space. Use

$$p_{j|i} = \frac{\exp(-d_{ij}^2 / 2\sigma_i^2)}{\sum_k \exp(-d_{ik}^2 / 2\sigma_i^2)}$$

**(a)** Compute all three $p_{j|i}$ for $\sigma_i = 1$.

**(b)** Compute the entropy $H = -\sum_j p_{j|i} \log_2 p_{j|i}$ (in bits) and the perplexity $\text{Perp} = 2^H$.

**(c)** For a dataset of $N = 10$ molecules, t-SNE symmetrizes the probabilities as $p_{ij} = (p_{j|i} + p_{i|j}) / 2N$. If $p_{j|i} = 0.45$ and $p_{i|j} = 0.25$, compute $p_{ij}$. Why is symmetrization needed?

**Your answer:**



### Part A2: t-SNE conditional probabilities

In [7]:
import numpy as np

# Given distances to neighbors
d_neighbors = np.array([1.0, 1.0, 2.0])
sigma_i = 1.0

# (a) Compute all three p_j|i for sigma_i = 1
# p_j|i = exp(-d_ij^2 / 2*sigma_i^2) / sum_k exp(-d_ik^2 / 2*sigma_i^2)

# Calculate numerator terms
numerators = np.exp(-d_neighbors**2 / (2 * sigma_i**2))

# Calculate sum of denominators
denominator = np.sum(numerators)

# Compute p_j|i
p_j_given_i = numerators / denominator

print(f"Distances d_ij: {d_neighbors}")
print(f"Sigma_i: {sigma_i}")
print(f"\nComputed p_j|i values: {p_j_given_i}")

# (b) Compute the entropy H = -sum_j p_j|i log2 p_j|i (in bits) and the perplexity Perp = 2^H

# Filter out p_j|i values that are very close to zero to avoid log(0)
# and potential NaN or inf issues. These values contribute negligibly to the sum anyway.
valid_p_j_given_i = p_j_given_i[p_j_given_i > 1e-10]

entropy = -np.sum(valid_p_j_given_i * np.log2(valid_p_j_given_i))
perplexity = 2**entropy

print(f"\nEntropy H: {entropy:.4f} bits")
print(f"Perplexity Perp: {perplexity:.4f}")

# (c) For a dataset of N = 10 molecules, t-SNE symmetrizes the probabilities as p_ij = (p_j|i + p_i|j) / 2N.
# If p_j|i = 0.45 and p_i|j = 0.25, compute p_ij. Why is symmetrization needed?

N = 10
p_j_given_i_val = 0.45
p_i_given_j_val = 0.25

p_ij = (p_j_given_i_val + p_i_given_j_val) / (2 * N)

print(f"\nGiven p_j|i = {p_j_given_i_val}, p_i|j = {p_i_given_j_val}, N = {N}")
print(f"Symmetrized probability p_ij = {p_ij:.4f}")

print("\nWhy is symmetrization needed? (Explanation):\n")
print("Symmetrization is crucial for t-SNE for several reasons:")
print("1. It ensures that the influence between two points i and j is mutual. Without symmetrization, p_j|i might be high while p_i|j is low, leading to inconsistencies in how points 'attract' each other in the low-dimensional embedding.")
print("2. It simplifies the gradient calculation during optimization, allowing t-SNE to use a single gradient term for each pair of points (i, j) rather than two separate terms.")
print("3. It prevents points from being 'orphan' or having a one-sided relationship, which could lead to artifacts in the embedding and distort the true underlying structure of the data.")
print("4. It ensures that the sum of all p_ij equals 1, which is required for p_ij to represent a proper probability distribution.")

Distances d_ij: [1. 1. 2.]
Sigma_i: 1.0

Computed p_j|i values: [0.44981622 0.44981622 0.10036756]

Entropy H: 1.3698 bits
Perplexity Perp: 2.5843

Given p_j|i = 0.45, p_i|j = 0.25, N = 10
Symmetrized probability p_ij = 0.0350

Why is symmetrization needed? (Explanation):

Symmetrization is crucial for t-SNE for several reasons:
1. It ensures that the influence between two points i and j is mutual. Without symmetrization, p_j|i might be high while p_i|j is low, leading to inconsistencies in how points 'attract' each other in the low-dimensional embedding.
2. It simplifies the gradient calculation during optimization, allowing t-SNE to use a single gradient term for each pair of points (i, j) rather than two separate terms.
3. It prevents points from being 'orphan' or having a one-sided relationship, which could lead to artifacts in the embedding and distort the true underlying structure of the data.
4. It ensures that the sum of all p_ij equals 1, which is required for p_ij to represen

In [ ]:
# Scratch: check A2 here
d = np.array([1.0, 1.0, 2.0])

## A3. Heavy tails

**(a)** At what map distance $d$ does the Student-t kernel $(1 + d^2)^{-1}$ fall to 10% of its value at $d = 0$?

**(b)** Answer the same question for the Gaussian kernel $\exp(-d^2)$.

**(c)** In one or two sentences, relate your answers to the crowding problem.

**Your answer:**



In [ ]:
# Scratch: check A3 here

### Part A3: Heavy tails

In [8]:
import numpy as np

# (a) At what map distance d does the Student-t kernel (1 + d^2)^-1 fall to 10% of its value at d = 0?
# Value at d = 0: (1 + 0^2)^-1 = 1
# We want (1 + d^2)^-1 = 0.1
# 1 + d^2 = 1 / 0.1 = 10
# d^2 = 9
# d = 3

d_student_t = np.sqrt(1/0.1 - 1)
print(f"For Student-t kernel, d = {d_student_t:.2f} when value falls to 10% of value at d=0")

# (b) Answer the same question for the Gaussian kernel exp(-d^2).
# Value at d = 0: exp(-0^2) = 1
# We want exp(-d^2) = 0.1
# -d^2 = ln(0.1)
# d^2 = -ln(0.1) = ln(10)
# d = sqrt(ln(10))

d_gaussian = np.sqrt(np.log(10))
print(f"For Gaussian kernel, d = {d_gaussian:.2f} when value falls to 10% of value at d=0")


For Student-t kernel, d = 3.00 when value falls to 10% of value at d=0
For Gaussian kernel, d = 1.52 when value falls to 10% of value at d=0


### (c) In one or two sentences, relate your answers to the crowding problem.

**Answer:** The Student-t kernel (t-SNE) falls off much slower (heavy tails) than the Gaussian kernel (PCA or other linear methods). This means that in the low-dimensional embedding, moderate distances in the original space can be mapped to much larger distances, which effectively alleviates the crowding problem by allowing distant points to be further apart, preventing them from being indistinguishable at the center of a cluster.

### Part A4: UMAP fuzzy weights

**(a) Show that the normalization condition reduces to $x + x^2 + x^3 = 1$, where $x = \exp(-0.5/\sigma_i)$.**

**Given:**
* $k = 4$ nearest neighbors with distances $d_1 = 0.5, d_2 = 1.0, d_3 = 1.5, d_4 = 2.0$.
* $\rho_i$ is the distance to the nearest neighbor, so $\rho_i = d_1 = 0.5$.
* UMAP fuzzy weight: $w_{ij} = \exp\left(-\frac{d_{ij} - \rho_i}{\sigma_i}\right)$
* Normalization condition: $\sum_j w_{ij} = \log_2 k$

**Derivation:**

Substitute $k=4$ into the normalization condition:
$\sum_j w_{ij} = \log_2 4 = 2$

Now, let's write out the sum for the four nearest neighbors:
$w_{i1} + w_{i2} + w_{i3} + w_{i4} = 2$

Substitute the formula for $w_{ij}$ and $\rho_i = 0.5$:
$\exp\left(-\frac{d_1 - 0.5}{\sigma_i}\right) + \exp\left(-\frac{d_2 - 0.5}{\sigma_i}\right) + \exp\left(-\frac{d_3 - 0.5}{\sigma_i}\right) + \exp\left(-\frac{d_4 - 0.5}{\sigma_i}\right) = 2$

Plug in the given distances:
$\exp\left(-\frac{0.5 - 0.5}{\sigma_i}\right) + \exp\left(-\frac{1.0 - 0.5}{\sigma_i}\right) + \exp\left(-\frac{1.5 - 0.5}{\sigma_i}\right) + \exp\left(-\frac{2.0 - 0.5}{\sigma_i}\right) = 2$

Simplify the exponents:
$\exp\left(-\frac{0}{\sigma_i}\right) + \exp\left(-\frac{0.5}{\sigma_i}\right) + \exp\left(-\frac{1.0}{\sigma_i}\right) + \exp\left(-\frac{1.5}{\sigma_i}\right) = 2$

Which simplifies to:
$1 + \exp\left(-\frac{0.5}{\sigma_i}\right) + \left(\exp\left(-\frac{0.5}{\sigma_i}\right)\right)^2 + \left(\exp\left(-\frac{0.5}{\sigma_i}\right)\right)^3 = 2$

Let $x = \exp(-0.5/\sigma_i)$. Substitute $x$ into the equation:
$1 + x + x^2 + x^3 = 2$

Rearranging the terms, we get:
$x + x^2 + x^3 = 1$

This completes the derivation for Part A4 (a).

In [9]:
# (b) Solve for x numerically and find sigma_i

def equation(x):
    return x**3 + x**2 + x - 1

# Bisection method to solve for x
def bisection_method(func, a, b, tol=1e-6, max_iter=100):
    if func(a) * func(b) >= 0:
        raise ValueError("Function has same sign at a and b.")

    for _ in range(max_iter):
        c = (a + b) / 2
        if abs(func(c)) < tol:
            return c
        if func(c) * func(a) < 0:
            b = c
        else:
            a = c
    return (a + b) / 2

# The function x^3 + x^2 + x - 1 has a root between 0 and 1
x_solution = bisection_method(equation, 0, 1)

print(f"\nNumerical solution for x: {x_solution:.4f}")

# Now find sigma_i from x = exp(-0.5/sigma_i)
# ln(x) = -0.5 / sigma_i
# sigma_i = -0.5 / ln(x)

sigma_i_umap = -0.5 / np.log(x_solution)
print(f"Calculated sigma_i: {sigma_i_umap:.4f}")


Numerical solution for x: 0.5437
Calculated sigma_i: 0.8205


In [10]:
# (c) If the weight from i to its second neighbor is wij and the reverse weight is wji = 0.5,
# compute the symmetrized weight using the fuzzy union w = a + b - ab.

# For the second neighbor, the distance is d2 = 1.0
# The weight from i to its second neighbor is w_i2 = exp(-(d2 - rho_i) / sigma_i)
# We have rho_i = 0.5
w_i2 = np.exp(-(1.0 - 0.5) / sigma_i_umap)

# Given w_ji = 0.5
w_ji = 0.5

# Symmetrized weight using fuzzy union: w = w_i2 + w_ji - w_i2 * w_ji
w_symmetrized = w_i2 + w_ji - (w_i2 * w_ji)

print(f"\nWeight from i to its second neighbor (w_i2): {w_i2:.4f}")
print(f"Given reverse weight (w_ji): {w_ji:.4f}")
print(f"Symmetrized weight using fuzzy union: {w_symmetrized:.4f}")


Weight from i to its second neighbor (w_i2): 0.5437
Given reverse weight (w_ji): 0.5000
Symmetrized weight using fuzzy union: 0.7718


## A4. UMAP fuzzy weights

For molecule $i$ with $k = 4$ nearest neighbors at distances 0.5, 1.0, 1.5, and 2.0, UMAP uses

$$w_{ij} = \exp\left(-\frac{d_{ij} - \rho_i}{\sigma_i}\right), \qquad \sum_j w_{ij} = \log_2 k$$

where $\rho_i$ is the distance to the nearest neighbor.

**(a)** Show that the normalization condition reduces to $x + x^2 + x^3 = 1$, where $x = \exp(-0.5/\sigma_i)$.

**(b)** Solve for $x$ numerically (for example by bisection) and find $\sigma_i$.

**(c)** If the weight from $i$ to its second neighbor is $w_{ij}$ and the reverse weight is $w_{ji} = 0.5$, compute the symmetrized weight using the fuzzy union $w = a + b - ab$.

**Your answer:**



In [ ]:
# Scratch: check A4 here (e.g. write a small bisection loop)

---
# Part B: Computational practice

The code below builds a dataset of about 90 small molecules in six chemical classes, computes 11 RDKit descriptors, and applies PCA, t-SNE, and UMAP. Run every cell in order, then answer the questions in the answer cells that follow them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
except ImportError:
    raise ImportError("RDKit is missing. Install it with:  pip install rdkit")

try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False
    print("umap-learn not found; UMAP cells will be skipped. Install with: pip install umap-learn")

SEED = 42

## 1. Build a small molecule dataset
Six chemical classes, generated systematically so every SMILES is valid.

In [ ]:
def build_dataset():
    rows = []

    def add(smiles, cls):
        rows.append({"smiles": smiles, "class": cls})

    for n in range(1, 11):
        add("C" * n + "O", "Alcohol")
    for m in range(1, 6):
        add("CC(O)" + "C" * m, "Alcohol")

    for n in range(1, 11):
        add("C" * n + "N", "Amine")
    for m in range(1, 6):
        add("CN(C)" + "C" * m, "Amine")

    for n in range(0, 10):
        add("C" * n + "C(=O)O", "Carboxylic acid")
    for m in range(1, 6):
        add("OC(=O)" + "C" * m + "C(=O)O", "Carboxylic acid")

    for n in range(1, 11):
        add("CC(=O)" + "C" * n, "Ketone")

    for n in range(1, 9):
        add("C" * n + "Cl", "Haloalkane")
    for n in range(1, 8):
        add("C" * n + "Br", "Haloalkane")

    for n in range(0, 10):
        add("c1ccccc1" + "C" * n, "Aromatic")
    for s in ["Cc1ccc(C)cc1", "Cc1cccc(C)c1", "c1ccc2ccccc2c1",
              "c1ccc(cc1)-c1ccccc1", "c1ccc2cc3ccccc3cc2c1"]:
        add(s, "Aromatic")

    return pd.DataFrame(rows)


def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "MolMR": Crippen.MolMR(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "RotBonds": Descriptors.NumRotatableBonds(mol),
        "Rings": rdMolDescriptors.CalcNumRings(mol),
        "AromRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FracCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
    }


df = build_dataset()
desc = pd.DataFrame([compute_descriptors(s) for s in df["smiles"]])
data = pd.concat([df, desc], axis=1)
feature_cols = list(desc.columns)
X_raw = data[feature_cols].to_numpy(dtype=float)

print(f"{len(data)} molecules, {len(feature_cols)} descriptors")
print(data["class"].value_counts())
data.head()

## 2. Plotting helper

In [ ]:
CLASSES = sorted(data["class"].unique())
MARKERS = ["o", "^", "s", "D", "v", "P"]
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]


def plot_by_class(ax, Y, title, xlabel="Dim 1", ylabel="Dim 2"):
    for cls, mk, col in zip(CLASSES, MARKERS, COLORS):
        idx = (data["class"] == cls).to_numpy()
        ax.scatter(Y[idx, 0], Y[idx, 1], marker=mk, c=col, s=35,
                   edgecolor="white", linewidth=0.5, label=cls)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


def plot_by_value(ax, Y, values, title, label):
    sc = ax.scatter(Y[:, 0], Y[:, 1], c=values, cmap="viridis", s=35,
                    edgecolor="white", linewidth=0.5)
    plt.colorbar(sc, ax=ax, label=label)
    ax.set_title(title, fontsize=10)

## 3. PCA: raw vs standardized descriptors

In [ ]:
X_std = StandardScaler().fit_transform(X_raw)

pca_raw = PCA().fit(X_raw - X_raw.mean(axis=0))
pca_std = PCA().fit(X_std)

print("Explained variance ratio (raw):         ", np.round(pca_raw.explained_variance_ratio_[:4], 4))
print("Explained variance ratio (standardized):", np.round(pca_std.explained_variance_ratio_[:4], 4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
Y_raw = pca_raw.transform(X_raw - X_raw.mean(axis=0))[:, :2]
Y_pca = pca_std.transform(X_std)[:, :2]
plot_by_class(axes[0], Y_raw, "PCA on raw descriptors", "PC1", "PC2")
plot_by_class(axes[1], Y_pca, "PCA on standardized descriptors", "PC1", "PC2")

k = np.arange(1, len(feature_cols) + 1)
axes[2].plot(k, np.cumsum(pca_raw.explained_variance_ratio_), "o-", label="raw")
axes[2].plot(k, np.cumsum(pca_std.explained_variance_ratio_), "s--", label="standardized")
axes[2].set_xlabel("Number of components")
axes[2].set_ylabel("Cumulative explained variance")
axes[2].set_title("Scree (cumulative)", fontsize=10)
axes[2].legend()
axes[1].legend(fontsize=8, loc="best")
plt.tight_layout()
plt.savefig("fig1_pca.png", dpi=150)
plt.show()

### PCA loadings (standardized)

In [17]:
loadings = pd.DataFrame(pca_std.components_[:2].T, index=feature_cols, columns=["PC1", "PC2"])
print(loadings.round(3).sort_values("PC1", key=abs, ascending=False))

              PC1    PC2
LogP        0.404  0.116
MolMR       0.389  0.239
HeavyAtoms  0.355  0.297
MolWt       0.348  0.295
Rings       0.319 -0.246
AromRings   0.319 -0.246
HBA        -0.252  0.360
HBD        -0.231  0.320
TPSA       -0.211  0.360
FracCSP3   -0.195  0.205
RotBonds    0.187  0.480


In [14]:
!pip install -q rdkit umap-learn

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
except ImportError:
    # RDKit should be installed by the setup cell, but this handles the case if it's not.
    raise ImportError("RDKit is missing. Please ensure the initial setup cells are run.")

def build_dataset():
    rows = []

    def add(smiles, cls):
        rows.append({"smiles": smiles, "class": cls})

    for n in range(1, 11):
        add("C" * n + "O", "Alcohol")
    for m in range(1, 6):
        add("CC(O)" + "C" * m, "Alcohol")

    for n in range(1, 11):
        add("C" * n + "N", "Amine")
    for m in range(1, 6):
        add("CN(C)" + "C" * m, "Amine")

    for n in range(0, 10):
        add("C" * n + "C(=O)O", "Carboxylic acid")
    for m in range(1, 6):
        add("OC(=O)" + "C" * m + "C(=O)O", "Carboxylic acid")

    for n in range(1, 11):
        add("CC(=O)" + "C" * n, "Ketone")

    for n in range(1, 9):
        add("C" * n + "Cl", "Haloalkane")
    for n in range(1, 8):
        add("C" * n + "Br", "Haloalkane")

    for n in range(0, 10):
        add("c1ccccc1" + "C" * n, "Aromatic")
    for s in ["Cc1ccc(C)cc1", "Cc1cccc(C)c1", "c1ccc2ccccc2c1",
              "c1ccc(cc1)-c1ccccc1", "c1ccc2cc3ccccc3cc2c1"]:
        add(s, "Aromatic")

    return pd.DataFrame(rows)


def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "MolMR": Crippen.MolMR(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "RotBonds": Descriptors.NumRotatableBonds(mol),
        "Rings": rdMolDescriptors.CalcNumRings(mol),
        "AromRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FracCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
    }

df = build_dataset()
desc = pd.DataFrame([compute_descriptors(s) for s in df["smiles"]])
data = pd.concat([df, desc], axis=1)
feature_cols = list(desc.columns)
X_raw = data[feature_cols].to_numpy(dtype=float)

pca_raw = PCA().fit(X_raw - X_raw.mean(axis=0))

loadings_raw = pd.DataFrame(pca_raw.components_[:2].T, index=feature_cols, columns=["PC1", "PC2"])
print("\nPCA Loadings (Raw Data - PC1 & PC2):\n")
print(loadings_raw.round(3).sort_values("PC1", key=abs, ascending=False))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 37.9 MB/s eta 0:00:00

PCA Loadings (Raw Data - PC1 & PC2):

              PC1    PC2
MolWt       0.949  0.067
MolMR       0.301 -0.083
HeavyAtoms  0.067  0.018
RotBonds    0.048  0.035
TPSA       -0.041  0.992
LogP        0.026 -0.026
Rings       0.004 -0.010
AromRings   0.004 -0.010
HBD        -0.003  0.028
HBA        -0.003  0.025
FracCSP3   -0.001  0.000


### YOUR TURN (B1)
1. Why does raw-data PCA give a very high PC1 explained variance? Which descriptor dominates?
2. Using the loadings, give a chemical name to PC1 and PC2 of the standardized PCA.
3. How many components are needed to reach 90% of the variance (standardized)?

**Your answer:**



In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_std = StandardScaler().fit_transform(X_raw)
pca_std = PCA().fit(X_std)

n90 = int(np.searchsorted(np.cumsum(pca_std.explained_variance_ratio_), 0.90) + 1)
print("Components needed for 90% variance (standardized):", n90)

Components needed for 90% variance (standardized): 3


## 4. t-SNE: effect of perplexity and random seed

In [ ]:
perplexities = [5, 15, 30, 50]
fig, axes = plt.subplots(1, len(perplexities), figsize=(18, 4.2))
tsne_results = {}
for ax, p in zip(axes, perplexities):
    Y = TSNE(n_components=2, perplexity=p, init="pca", learning_rate="auto",
             random_state=SEED).fit_transform(X_std)
    tsne_results[p] = Y
    plot_by_class(ax, Y, f"t-SNE, perplexity = {p}", "t-SNE 1", "t-SNE 2")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig("fig2_tsne_perplexity.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, seed in zip(axes, [0, 1, 2]):
    Y = TSNE(n_components=2, perplexity=15, init="random", learning_rate="auto",
             random_state=seed).fit_transform(X_std)
    plot_by_class(ax, Y, f"t-SNE, perplexity = 15, seed = {seed}", "t-SNE 1", "t-SNE 2")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig("fig3_tsne_seeds.png", dpi=150)
plt.show()

### YOUR TURN (B2)
1. How does the map change as perplexity increases from 5 to 50?
2. Which features stay the same across seeds, and which change?
3. Why is perplexity = 100 not allowed for this dataset?

**Your answer:**



## 5. UMAP: effect of n_neighbors and min_dist

In [ ]:
umap_results = {}
if HAVE_UMAP:
    settings = [(5, 0.1), (15, 0.1), (50, 0.1), (15, 0.8)]
    fig, axes = plt.subplots(1, len(settings), figsize=(18, 4.2))
    for ax, (nn, md) in zip(axes, settings):
        Y = umap.UMAP(n_neighbors=nn, min_dist=md, random_state=SEED).fit_transform(X_std)
        umap_results[(nn, md)] = Y
        plot_by_class(ax, Y, f"UMAP, n_neighbors = {nn}, min_dist = {md}", "UMAP 1", "UMAP 2")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig("fig4_umap.png", dpi=150)
    plt.show()

### YOUR TURN (B3)
1. Compare n_neighbors = 5 and 50. Which shows more local detail, which more global layout?
2. What does increasing min_dist from 0.1 to 0.8 do to the clusters?

**Your answer:**



## 6. Colour by a continuous property

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
plot_by_value(axes[0], Y_pca, data["LogP"], "PCA coloured by LogP", "LogP")
plot_by_value(axes[1], tsne_results[30], data["LogP"], "t-SNE coloured by LogP", "LogP")
if HAVE_UMAP:
    plot_by_value(axes[2], umap_results[(15, 0.1)], data["LogP"], "UMAP coloured by LogP", "LogP")
else:
    axes[2].axis("off")
plt.tight_layout()
plt.savefig("fig5_logp.png", dpi=150)
plt.show()

### YOUR TURN (B4)
Inside each class cluster, is there a smooth LogP gradient? What structural change
(hint: chain length) produces it?

**Your answer:**



## 7. Quantitative comparison: trustworthiness
Trustworthiness (0 to 1) measures how well each point's nearest neighbours in the map
were also neighbours in the original space. 1 means perfect local preservation.

In [ ]:
emb = {"PCA (2 PCs)": Y_pca, "t-SNE (perp 30)": tsne_results[30]}
if HAVE_UMAP:
    emb["UMAP (nn 15)"] = umap_results[(15, 0.1)]

for k_nn in [5, 15]:
    for name, Y in emb.items():
        t = trustworthiness(X_std, Y, n_neighbors=k_nn)
        print(f"k = {k_nn:2d}  {name:18s} trustworthiness = {t:.3f}")

### YOUR TURN (B5)
1. Which method preserves local neighbourhoods best? Is that what you expected?
2. Trustworthiness says nothing about distances between clusters. Propose a check
   for global structure (hint: correlate pairwise distances in the original and 2D spaces).

**Your answer:**


In [21]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from sklearn.manifold import TSNE
import umap # Ensure umap is imported if HAVE_UMAP is True

SEED = 42 # Define SEED for reproducibility

try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False
    print("umap-learn not found; UMAP cells will be skipped.")

# Re-compute Y_pca (from X7gkY3rLWngd)
Y_pca = pca_std.transform(X_std)[:, :2]

# Re-compute tsne_results[30] (from cOjW3g5YWngg)
tsne_results = {}
p = 30
Y_tsne_30 = TSNE(n_components=2, perplexity=p, init="pca", learning_rate="auto",
         random_state=SEED).fit_transform(X_std)
tsne_results[p] = Y_tsne_30

# Re-compute umap_results[(15, 0.1)] (from _tdicn-xWngj)
umap_results = {}
if HAVE_UMAP:
    nn, md = 15, 0.1
    Y_umap_15_01 = umap.UMAP(n_neighbors=nn, min_dist=md, random_state=SEED).fit_transform(X_std)
    umap_results[(nn, md)] = Y_umap_15_01

def global_structure_check(X_high_dim, Y_low_dim, name):
    # Calculate pairwise distances in high-dimensional space
    dist_high_dim = pdist(X_high_dim)

    # Calculate pairwise distances in low-dimensional space
    dist_low_dim = pdist(Y_low_dim)

    # Compute Spearman's rank correlation coefficient
    correlation, _ = spearmanr(dist_high_dim, dist_low_dim)
    print(f"Spearman correlation for {name} (global structure): {correlation:.3f}")

print("\nGlobal Structure Check (Spearman's Rank Correlation of Pairwise Distances):\n")

global_structure_check(X_std, Y_pca, "PCA (2 PCs)")
global_structure_check(X_std, tsne_results[30], "t-SNE (perp 30)")

if HAVE_UMAP:
    global_structure_check(X_std, umap_results[(15, 0.1)], "UMAP (nn 15)")

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Global Structure Check (Spearman's Rank Correlation of Pairwise Distances):

Spearman correlation for PCA (2 PCs) (global structure): 0.954
Spearman correlation for t-SNE (perp 30) (global structure): 0.866
Spearman correlation for UMAP (nn 15) (global structure): 0.756
